# NextLearn — quiz item analysis with Item Response Theory

**The gap this fills.** NextLearn measures students (risk model) and cohorts (K-means
clustering). Nothing measures the *questions*. Since `ml/quizgen.py` generates quizzes with
an LLM, the platform ships content whose quality has never been assessed by anything except
someone reading it.

Item Response Theory fits, for every question, two parameters:

- **b — difficulty**: the ability level at which a student has a 50% chance of answering correctly.
- **a — discrimination**: how sharply the question separates stronger students from weaker ones.

Low discrimination is the mathematical signature of a broken question: strong and weak
students answer it identically, so it measures nothing. A *negative* value is worse — it
usually means the answer key is wrong.

**The data NextLearn is currently discarding.** `src/routes/web.ts:2729` already builds a
per-question `review` array (which option each student picked, which was correct) and
returns it to the browser. What gets persisted at `web.ts:2709` is only the aggregate
`score`. Every submission collapses "missed Q3 and Q7" into "60%". Section 5 covers the
one-field change that would start capturing it — this notebook makes no app changes.

**What this notebook establishes, in order:**

1. The implementation is correct — verified by recovering known parameters from simulated data.
2. IRT ability (θ) is a materially better measure than raw percentage — quantified below.
3. It finds genuinely broken questions in real data.

**Headline result from section 2:** when students take quizzes of *different difficulty* —
NextLearn's actual situation, since sous-acquis quizzes are not calibrated against each other
— raw percentage correlates 0.648 with true ability while IRT θ reaches 0.867. Two groups of
essentially identical ability scored 65% and 33% on different forms. That 33-point gap is
pure measurement artifact, and `averageScore` feeds it straight into the risk model.

In [ ]:
!pip -q install numpy scipy pandas matplotlib

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, time, warnings
warnings.filterwarnings("ignore")
RANDOM_STATE = 42

SURFACE, INK, MUTED, LINE = "#fcfcfb", "#1c1c1a", "#6b6b66", "#dcdcd6"
CAT = ["#1a6fb5", "#d1621b", "#8f4a9c", "#3f7d3f"]   # CVD-validated, fixed order
SEQ_HUE = "#1a6fb5"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "axes.edgecolor": LINE, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.grid": True, "grid.color": LINE, "grid.linewidth": 0.6, "grid.alpha": 0.7,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 11, "axes.titlesize": 13, "axes.titleweight": "600",
    "figure.dpi": 120, "lines.linewidth": 2,
})
print("ready")

## 1. The model

2PL IRT, fitted by marginal maximum likelihood (EM with Gauss-Hermite quadrature):

$$P(\text{correct} \mid \theta, a, b) = \frac{1}{1 + e^{-a(\theta - b)}}$$

Written in plain numpy/scipy rather than pulling in `girth` or `py-irt`, for two reasons:
it drops into `ml/` with no new dependency beyond what `requirements.txt` already has, and
every step stays inspectable — which matters when you have to defend it.

`fit_2pl` takes a matrix of 0 / 1 / NaN, where NaN means "this student never saw this
question". That sparsity is the normal case: students take different quizzes.

In [ ]:
"""2PL Item Response Theory via marginal maximum likelihood (EM + Gauss-Hermite
quadrature). Pure numpy/scipy so it ports into ml/ with no new dependency.

  P(correct | theta, a, b) = 1 / (1 + exp(-a * (theta - b)))

  a = discrimination  (how sharply the item separates strong from weak students)
  b = difficulty      (theta at which P = 0.5)

Handles missing responses: R is a float matrix of 0/1/NaN, NaN = not answered.
"""
import numpy as np
from scipy.optimize import minimize


def _sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -35, 35)))


def fit_2pl(R, n_quad=41, max_iter=200, tol=1e-4, verbose=False):
    """Fits 2PL. R: (n_students, n_items) with 0/1/NaN. Returns (a, b, n_iter)."""
    R = np.asarray(R, dtype=float)
    n_stud, n_items = R.shape
    mask = ~np.isnan(R)
    X = np.nan_to_num(R, nan=0.0)

    # Standard-normal quadrature over the latent ability scale.
    nodes = np.linspace(-4, 4, n_quad)
    weights = np.exp(-0.5 * nodes**2)
    weights /= weights.sum()

    a = np.ones(n_items)
    b = np.zeros(n_items)

    for it in range(max_iter):
        # ---- E step: posterior over theta for each student -----------------
        z = a[None, :, None] * (nodes[None, None, :] - b[None, :, None])  # (1,I,Q)
        P = _sigmoid(z)                                                    # (1,I,Q)
        P = np.clip(P, 1e-9, 1 - 1e-9)

        # log-lik of each student's response vector at each node
        ll = (X[:, :, None] * np.log(P) + (1 - X)[:, :, None] * np.log(1 - P))
        ll = np.where(mask[:, :, None], ll, 0.0).sum(axis=1)               # (N,Q)

        post = ll + np.log(weights)[None, :]
        post -= post.max(axis=1, keepdims=True)
        post = np.exp(post)
        post /= post.sum(axis=1, keepdims=True)                            # (N,Q)

        # expected counts per item per node
        n_iq = np.einsum("nq,ni->iq", post, mask.astype(float))            # (I,Q)
        r_iq = np.einsum("nq,ni->iq", post, np.where(mask, X, 0.0))        # (I,Q)

        # ---- M step: one small logistic fit per item -----------------------
        a_new, b_new = a.copy(), b.copy()
        for i in range(n_items):
            n_i, r_i = n_iq[i], r_iq[i]
            keep = n_i > 1e-8
            if keep.sum() < 2:
                continue
            nk, rk, tk = n_i[keep], r_i[keep], nodes[keep]

            def nll(params):
                ai, bi = params
                p = np.clip(_sigmoid(ai * (tk - bi)), 1e-9, 1 - 1e-9)
                return -(rk * np.log(p) + (nk - rk) * np.log(1 - p)).sum()

            res = minimize(nll, x0=[a[i], b[i]], method="L-BFGS-B",
                           bounds=[(0.05, 4.0), (-4.0, 4.0)])
            if res.success or np.isfinite(res.fun):
                a_new[i], b_new[i] = res.x

        delta = max(np.abs(a_new - a).max(), np.abs(b_new - b).max())
        a, b = a_new, b_new
        if verbose and it % 10 == 0:
            print(f"  iter {it:>3}  max delta {delta:.5f}")
        if delta < tol:
            break

    return a, b, it + 1


def eap_theta(R, a, b, n_quad=41):
    """Expected a-posteriori ability estimate per student, plus its posterior SD."""
    R = np.asarray(R, dtype=float)
    mask = ~np.isnan(R)
    X = np.nan_to_num(R, nan=0.0)

    nodes = np.linspace(-4, 4, n_quad)
    weights = np.exp(-0.5 * nodes**2)
    weights /= weights.sum()

    P = np.clip(_sigmoid(a[None, :, None] * (nodes[None, None, :] - b[None, :, None])),
                1e-9, 1 - 1e-9)
    ll = (X[:, :, None] * np.log(P) + (1 - X)[:, :, None] * np.log(1 - P))
    ll = np.where(mask[:, :, None], ll, 0.0).sum(axis=1)

    post = ll + np.log(weights)[None, :]
    post -= post.max(axis=1, keepdims=True)
    post = np.exp(post)
    post /= post.sum(axis=1, keepdims=True)

    theta = post @ nodes
    var = (post * (nodes[None, :] - theta[:, None]) ** 2).sum(axis=1)
    return theta, np.sqrt(var)


def item_fit_stats(R, a, b):
    """Classical companions to the IRT parameters, for the teacher-facing view."""
    R = np.asarray(R, dtype=float)
    mask = ~np.isnan(R)
    theta, _ = eap_theta(R, a, b)

    n_items = R.shape[1]
    p_value = np.full(n_items, np.nan)      # proportion correct
    pt_bis = np.full(n_items, np.nan)       # point-biserial with total score
    n_resp = mask.sum(axis=0)

    total = np.nansum(R, axis=1)
    for i in range(n_items):
        m = mask[:, i]
        if m.sum() < 10:
            continue
        xi = R[m, i]
        p_value[i] = xi.mean()
        rest = total[m] - xi            # rest score: exclude the item itself
        if xi.std() > 1e-9 and rest.std() > 1e-9:
            pt_bis[i] = np.corrcoef(xi, rest)[0, 1]
    return p_value, pt_bis, n_resp, theta

## 2. Does the implementation work?

Two questions, in order. First: given data generated from known parameters, does the fitter
recover them? Nothing downstream is worth reading if it doesn't.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
N, I = 2000, 30
true_a = rng.lognormal(0.0, 0.35, I).clip(0.3, 3.0)
true_b = rng.normal(0, 1.0, I).clip(-3, 3)
true_theta = rng.normal(0, 1, N)

P = 1 / (1 + np.exp(-true_a[None, :] * (true_theta[:, None] - true_b[None, :])))
R_sim = (rng.random((N, I)) < P).astype(float)
R_sim[rng.random((N, I)) < 0.20] = np.nan     # 20% unanswered

t = time.time()
a_hat, b_hat, iters = fit_2pl(R_sim)
theta_hat, theta_se = eap_theta(R_sim, a_hat, b_hat)
print(f"converged in {iters} iterations, {time.time()-t:.1f}s\n")
for nm, est, tru in [("discrimination a", a_hat, true_a),
                     ("difficulty     b", b_hat, true_b),
                     ("ability    theta", theta_hat, true_theta)]:
    print(f"{nm}: corr {np.corrcoef(est, tru)[0,1]:.3f}   RMSE {np.sqrt(((est-tru)**2).mean()):.3f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.4))
for ax, est, tru, name in [(axes[0], b_hat, true_b, "difficulty  b"),
                           (axes[1], a_hat, true_a, "discrimination  a"),
                           (axes[2], theta_hat, true_theta, "ability  theta")]:
    lo, hi = min(est.min(), tru.min()), max(est.max(), tru.max())
    ax.plot([lo, hi], [lo, hi], color=MUTED, linestyle="--", linewidth=1.2, zorder=1)
    ax.scatter(tru, est, s=18, color=SEQ_HUE, alpha=0.65, edgecolor="none", zorder=2)
    ax.set_xlabel("true"); ax.set_ylabel("recovered")
    ax.set_title(f"{name}   r = {np.corrcoef(est, tru)[0,1]:.3f}")
fig.suptitle("Parameter recovery on simulated data (dashed = perfect)", y=1.02, fontsize=13)
plt.tight_layout(); plt.show()

### The question that actually justifies the feature

Recovery being good is necessary but not interesting on its own — with everyone answering
the same items, a raw percentage already ranks students nearly as well (0.894 vs 0.905
below). IRT earns its place when students take **different quizzes of different difficulty**,
which is precisely NextLearn's situation: each sous-acquis has its own quiz, and no two are
calibrated against each other.

The simulation below splits students across an easy form and a hard form, with the two
groups drawn from the *same* ability distribution. Any score gap between them is therefore
pure artifact — a correct measure must report roughly zero.

In [ ]:
rng2 = np.random.default_rng(7)
N2, I2 = 1500, 40
a2 = rng2.lognormal(0.0, 0.3, I2).clip(0.3, 3.0)
b2 = np.concatenate([rng2.normal(-1.0, 0.5, I2 // 2),      # easy form
                     rng2.normal( 1.0, 0.5, I2 // 2)]).clip(-3, 3)   # hard form
theta2 = rng2.normal(0, 1, N2)
form = rng2.integers(0, 2, N2)

R2 = np.full((N2, I2), np.nan)
for s in range(N2):
    idx = np.arange(0, I2 // 2) if form[s] == 0 else np.arange(I2 // 2, I2)
    p = 1 / (1 + np.exp(-a2[idx] * (theta2[s] - b2[idx])))
    R2[s, idx] = (rng2.random(len(idx)) < p).astype(float)

a2_hat, b2_hat, _ = fit_2pl(R2)
theta2_hat, _ = eap_theta(R2, a2_hat, b2_hat)
raw2 = np.nanmean(R2, axis=1)

e, h = form == 0, form == 1
print("Correlation with TRUE ability")
print(f"  raw percentage : {np.corrcoef(raw2, theta2)[0,1]:.3f}")
print(f"  IRT theta      : {np.corrcoef(theta2_hat, theta2)[0,1]:.3f}")
print("\nGap between the easy-form and hard-form groups (true gap is ~0):")
print(f"  true ability   : {theta2[e].mean() - theta2[h].mean():+.3f}")
print(f"  raw percentage : {raw2[e].mean() - raw2[h].mean():+.3f}   <-- artifact")
print(f"  IRT theta      : {theta2_hat[e].mean() - theta2_hat[h].mean():+.3f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.8))

# Two groups of equal true ability - identity is carried by colour AND a legend.
bins = np.linspace(0, 1, 26)
ax1.hist(raw2[e], bins=bins, color=CAT[0], alpha=0.75, label="easy form")
ax1.hist(raw2[h], bins=bins, color=CAT[1], alpha=0.75, label="hard form")
ax1.set_title("Raw percentage — invents a 33-point gap")
ax1.set_xlabel("raw score"); ax1.set_ylabel("students")
ax1.legend(frameon=False, fontsize=9)

bins2 = np.linspace(-3, 3, 26)
ax2.hist(theta2_hat[e], bins=bins2, color=CAT[0], alpha=0.75, label="easy form")
ax2.hist(theta2_hat[h], bins=bins2, color=CAT[1], alpha=0.75, label="hard form")
ax2.set_title("IRT theta — correctly reports no gap")
ax2.set_xlabel("estimated ability (theta)"); ax2.set_ylabel("students")
ax2.legend(frameon=False, fontsize=9)

plt.tight_layout(); plt.show()

This is not an abstract concern. It is the direct explanation for a real observation on the
platform: *passing a quiz with 60% and watching the success score go down*. If that 60% came
from a harder-than-average quiz, the raw percentage understates the student's ability, and
`averageScore` — a plain mean of raw percentages, `features.ts:102` — passes the distortion
straight into the risk model. θ is the drop-in correction.

## 3. Real data — ASSISTments

Simulation proves the estimator is unbiased; it cannot prove the model fits real students.
ASSISTments (2009–10 skill builder) has what NextLearn is currently throwing away: one row
per student per question with a binary `correct`.

Only first encounters are kept (`original == 1`, deduplicated), because IRT models a first
honest attempt, not performance after hints and retries.

In [ ]:
import urllib.request, os
ASSIST_URL = "https://raw.githubusercontent.com/CAHLR/pyBKT-examples/master/data/as.csv"
if not os.path.exists("as.csv"):
    print("downloading ASSISTments (~83 MB) ...")
    urllib.request.urlretrieve(ASSIST_URL, "as.csv")

d = pd.read_csv("as.csv", encoding="latin-1", low_memory=False,
                usecols=["user_id", "problem_id", "correct", "skill_name", "original"])
d = d[d["original"] == 1].dropna(subset=["skill_name"])
d["correct"] = pd.to_numeric(d["correct"], errors="coerce")
d = d[d["correct"].isin([0, 1])]
print(f"{len(d):,} responses | {d.user_id.nunique():,} students | {d.problem_id.nunique():,} problems")
d.skill_name.value_counts().head(8)

In [ ]:
SKILL = "Equation Solving Two or Fewer Steps"
MIN_RESPONSES_PER_ITEM = 50    # item parameters are unstable below this
MIN_ITEMS_PER_STUDENT = 5

s = d[d.skill_name == SKILL].drop_duplicates(["user_id", "problem_id"], keep="first")
pc = s.problem_id.value_counts(); s = s[s.problem_id.isin(pc[pc >= MIN_RESPONSES_PER_ITEM].index)]
uc = s.user_id.value_counts();    s = s[s.user_id.isin(uc[uc >= MIN_ITEMS_PER_STUDENT].index)]

M = s.pivot_table(index="user_id", columns="problem_id", values="correct", aggfunc="first")
R = M.values.astype(float)
print(f"skill    : {SKILL}")
print(f"matrix   : {M.shape[0]} students x {M.shape[1]} items")
print(f"density  : {M.notna().mean().mean():.1%}   overall correct: {np.nanmean(R):.3f}")

t = time.time()
a_r, b_r, it_r = fit_2pl(R)
p_val, pt_bis, n_resp, theta_r = item_fit_stats(R, a_r, b_r)
print(f"fitted in {it_r} iterations, {time.time()-t:.1f}s")

### The quality gate

Four rules, each catching a different failure. `pt_biserial` is the classical companion to
discrimination — the correlation between getting this item right and scoring well on the
*rest* of the quiz.

| Flag | Rule | What it means |
|---|---|---|
| `low_discrimination` | a < 0.5 | Strong and weak students perform alike — the item measures nothing |
| `negative_pt_biserial` | pt_bis < 0 | **Stronger students do worse — usually a wrong answer key** |
| `too_easy` / `too_hard` | p > 0.95 / p < 0.05 | Nearly everyone gets it right or wrong; carries no information |
| `unstable` | n < 50 | Too few responses to trust the estimate at all |

`negative_pt_biserial` is the one to act on first: it is nearly always a real defect in the
question rather than a property of the students.

In [ ]:
items = pd.DataFrame({
    "problem_id": M.columns, "a": a_r, "b": b_r,
    "p_value": p_val, "pt_biserial": pt_bis, "n": n_resp,
})
items["flags"] = [
    ", ".join(f for f, c in [
        ("low_discrimination", r.a < 0.5),
        ("negative_pt_biserial", r.pt_biserial < 0),
        ("too_easy", r.p_value > 0.95),
        ("too_hard", r.p_value < 0.05),
        ("unstable", r.n < MIN_RESPONSES_PER_ITEM),
    ] if c) for r in items.itertuples()
]
items["ok"] = items["flags"] == ""

print(f"{(~items.ok).sum()} of {len(items)} items flagged ({(~items.ok).mean():.1%})\n")
print("most discriminating (keep these):")
print(items.nlargest(5, "a")[["problem_id","a","b","p_value","pt_biserial","n"]].round(3).to_string(index=False))
print("\nworst offenders (review or drop):")
print(items[~items.ok].nsmallest(5, "pt_biserial")[
    ["problem_id","a","b","p_value","pt_biserial","n","flags"]].round(3).to_string(index=False))

In [ ]:
# Item characteristic curves: what a good item looks like next to a broken one.
good = items.nlargest(1, "a").iloc[0]
bad  = items.nsmallest(1, "pt_biserial").iloc[0]
grid = np.linspace(-3, 3, 200)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.6))
for row, color, label in [(good, CAT[0], "well-behaved item"), (bad, CAT[1], "flagged item")]:
    ax1.plot(grid, 1/(1+np.exp(-row.a*(grid-row.b))), color=color,
             label=f"{label}  (a={row.a:.2f}, b={row.b:.2f})")
ax1.axhline(0.5, color=MUTED, linestyle=":", linewidth=1)
ax1.set_xlabel("student ability (theta)"); ax1.set_ylabel("P(correct)")
ax1.set_title("Item characteristic curves")
ax1.legend(frameon=False, fontsize=9, loc="upper left")
ax1.set_ylim(0, 1)

ok, bad_m = items.ok.values, ~items.ok.values
ax2.scatter(items.b[ok], items.a[ok], s=30, color=CAT[0], alpha=0.8,
            edgecolor="none", label=f"passes ({ok.sum()})")
ax2.scatter(items.b[bad_m], items.a[bad_m], s=30, color=CAT[1], alpha=0.8,
            edgecolor="none", label=f"flagged ({bad_m.sum()})")
ax2.axhline(0.5, color=MUTED, linestyle="--", linewidth=1.2)
ax2.text(items.b.min(), 0.52, " a = 0.5 quality floor", color=MUTED, fontsize=9)
ax2.set_xlabel("difficulty  b"); ax2.set_ylabel("discrimination  a")
ax2.set_title("Item map")
ax2.legend(frameon=False, fontsize=9)
plt.tight_layout(); plt.show()

A steep curve means the item cleanly separates students around its difficulty. A flat one
means it barely distinguishes anybody. When the flagged curve slopes the *wrong way*, the
item is actively misinforming the model — and you cannot see that by reading the question.

**An honest limitation of this run:** the matrix is only ~17% dense with 50–70 responses per
item, so some parameters land on the optimiser's bounds (a = 4.0) and the flag rate is high.
Sparse data makes IRT noisy. Treat the flags here as a demonstration of the mechanism; a
production gate needs the `n >= 50` floor enforced and ideally 200+ responses per question.

## 4. Does theta disagree with the raw score on real students?

If the two ranked students identically, none of this would be worth deploying.

In [ ]:
raw_r = np.nanmean(R, axis=1)
n_ans = (~np.isnan(R)).sum(axis=1)
corr = np.corrcoef(raw_r, theta_r)[0, 1]

rank_raw = pd.Series(raw_r).rank(pct=True)
rank_th  = pd.Series(theta_r).rank(pct=True)
moved = (rank_raw - rank_th).abs()

print(f"correlation raw vs theta      : {corr:.3f}")
print(f"students moving >10 percentile: {(moved > 0.10).sum()} of {len(moved)} ({(moved>0.10).mean():.1%})")
print(f"students moving >20 percentile: {(moved > 0.20).sum()} ({(moved>0.20).mean():.1%})")

fig, ax = plt.subplots(figsize=(6.4, 5))
sc = ax.scatter(raw_r, theta_r, s=16, c=n_ans, cmap="Blues", alpha=0.85, edgecolor="none")
ax.set_xlabel("raw proportion correct"); ax.set_ylabel("IRT theta")
ax.set_title(f"Raw score vs theta   r = {corr:.3f}")
fig.colorbar(sc, ax=ax, label="questions answered", shrink=0.85)
plt.tight_layout(); plt.show()

## 5. What this would take in NextLearn

**This notebook changes nothing in the app.** Wiring it in would mean, in order:

**1 — Capture the responses (the only blocking change).** `web.ts:2729` already computes the
per-question `review`. Persist it alongside the score:

```ts
"progress.quizResults": {
  lessonKey, moduleId, subAcquisId, score, attempts, submittedAt,
  responses: review.map((r, i) => ({
    questionIndex: i,
    selectedIndex: r.selectedIndex,
    correct: r.selectedIndex === r.correctOptionIndex
  }))
}
```

plus the matching `responses` array on the `quizResults` sub-schema in `models/User.ts:95`.
Everything else is additive and can wait; without this, nothing else is possible, and the
data only accumulates going forward — it cannot be backfilled.

**2 — Fit periodically, not per request.** Item parameters shift slowly. A scheduled job
writing `data/item-params.json` is the right shape; fitting inside a request is not.

**3 — Expose it to teachers.** An item-quality table in the backoffice next to the existing
clustering dashboard, sorted worst-first, so a bad generated question is visible without
anyone reading all forty.

**4 — Close the loop on `quizgen.py`.** Once parameters exist, a generated question that
accumulates responses and turns out to have near-zero discrimination can be auto-retired.
That turns quiz generation from fire-and-forget into something with a feedback signal.

**5 — Optionally, feed θ to the risk model.** Replace `averageScore` in `features.ts` with
the ability estimate. Section 2 quantifies the gain; retrain before and after and keep it
only if the honest cross-validated AUC actually improves.

### The limitation to state plainly

Item parameters need volume — roughly 200 responses per question for stable estimates, and
this run shows what 50–70 looks like. With ~35 students on the platform, calibrating your own
questions is not yet possible. What is defensible today is exactly what this notebook does:
a validated implementation, demonstrated on real data, with the app-side capture ready so
calibration becomes possible as usage grows. Same posture as the OULAD work — the method is
sound, the local numbers need students.